# Pothole + Patching Detection — YOLOv11 Training

**Before running:** `Runtime → Change runtime type → T4 GPU`

| Cell | Does |
|------|------|
| 1 | Install dependencies |
| 2 | Verify GPU is active |
| 3 | Mount Drive & verify dataset |
| 4 | Convert annotations + Train |
| 5 | Save model to Drive |

In [ ]:
# ─── Cell 1: Install dependencies ───────────────────────────────────────────
!pip install -q ultralytics

In [ ]:
# ─── Cell 2: Verify GPU ──────────────────────────────────────────────────────
# STOP if this cell shows 'No GPU'. Go to Runtime → Change runtime type → T4 GPU

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected!\n"
        "Go to Runtime → Change runtime type → Hardware accelerator → T4 GPU"
    )

gpu_name  = torch.cuda.get_device_name(0)
vram_gb   = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU   : {gpu_name}")
print(f"VRAM  : {vram_gb:.1f} GB")
print(f"CUDA  : {torch.version.cuda}")
print(f"FP16  : {torch.cuda.is_bf16_supported() or True}  (T4 supports FP16 Tensor Cores)")
print("\nGPU ready!")

In [ ]:
# ─── Cell 3: Mount Google Drive & verify dataset ─────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

INDIA_DRIVE_PATH = '/content/drive/MyDrive/India'   # ← change if needed

assert Path(INDIA_DRIVE_PATH).exists(), (
    f"Dataset not found at: {INDIA_DRIVE_PATH}\n"
    "Upload the 'India' folder to Google Drive and update INDIA_DRIVE_PATH."
)

n_train = len(list(Path(INDIA_DRIVE_PATH, 'train', 'images').glob('*.jpg')))
n_xml   = len(list(Path(INDIA_DRIVE_PATH, 'train', 'annotations', 'xmls').glob('*.xml')))
print(f'Train images  : {n_train}')
print(f'Annotations   : {n_xml}')
print('Dataset OK!')

In [ ]:
# ─── Cell 4: Convert annotations + Train (PARALLEL COPY) ────────────────────

from __future__ import annotations
import random, shutil, xml.etree.ElementTree as ET, yaml
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
from ultralytics import YOLO

# ── Configuration ──────────────────────────────────────────────────────────
CLASS_MAP = {
    'D40': 0,   # Pothole
    'D11': 0,   # Pothole
    'D20': 1,   # Patching
    'D44': 1,   # Patching
}
CLASS_NAMES = ['Pothole', 'Patching']

DATASET_DIR = Path(INDIA_DRIVE_PATH)
OUTPUT_DIR  = Path('/content/yolo_dataset')

YOLO_MODEL  = 'yolo11m.pt'
EPOCHS      = 100
IMG_SIZE    = 640
BATCH_SIZE  = 16
VAL_SPLIT   = 0.10
RANDOM_SEED = 42
COPY_THREADS = 32   # Drive handles 32 parallel reads well

# ── Helpers ────────────────────────────────────────────────────────────────
def voc_to_yolo(w, h, xmin, ymin, xmax, ymax):
    return (xmin+xmax)/2/w, (ymin+ymax)/2/h, (xmax-xmin)/w, (ymax-ymin)/h

def convert_xml(xml_path, label_path):
    root = ET.parse(xml_path).getroot()
    size = root.find('size')
    if size is None: return 0
    W, H = int(size.find('width').text), int(size.find('height').text)
    if W == 0 or H == 0: return 0
    lines = []
    for obj in root.findall('object'):
        name = obj.find('name').text.strip()
        if name not in CLASS_MAP: continue
        bb = obj.find('bndbox')
        xc, yc, bw, bh = voc_to_yolo(W, H,
            float(bb.find('xmin').text), float(bb.find('ymin').text),
            float(bb.find('xmax').text), float(bb.find('ymax').text))
        lines.append(f'{CLASS_MAP[name]} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}')
    label_path.parent.mkdir(parents=True, exist_ok=True)
    label_path.write_text('\n'.join(lines))
    return len(lines)

# ── Plan splits ────────────────────────────────────────────────────────────
print('[1/3] Planning train/val split ...')
xml_dir    = DATASET_DIR / 'train' / 'annotations' / 'xmls'
img_dir    = DATASET_DIR / 'train' / 'images'
all_images = sorted(img_dir.glob('*.jpg'))
random.seed(RANDOM_SEED)
random.shuffle(all_images)
val_stems = {p.stem for p in all_images[:int(len(all_images) * VAL_SPLIT)]}

# Pre-create all destination directories
for split in ('train', 'val'):
    (OUTPUT_DIR / split / 'images').mkdir(parents=True, exist_ok=True)
    (OUTPUT_DIR / split / 'labels').mkdir(parents=True, exist_ok=True)

def process_one(img: Path):
    """Copy image + write YOLO label. Runs in a worker thread."""
    split     = 'val' if img.stem in val_stems else 'train'
    dst_img   = OUTPUT_DIR / split / 'images' / img.name
    dst_label = OUTPUT_DIR / split / 'labels' / img.with_suffix('.txt').name

    if not dst_img.exists():
        shutil.copy2(img, dst_img)   # the slow call — many of these run in parallel

    xml = xml_dir / img.with_suffix('.xml').name
    if xml.exists():
        n = convert_xml(xml, dst_label)
    else:
        dst_label.write_text('')
        n = 0
    return split, n

# ── Parallel copy + convert ────────────────────────────────────────────────
print(f'  Copying {len(all_images)} images with {COPY_THREADS} parallel threads ...')
stats = {'train': [0, 0], 'val': [0, 0]}
with ThreadPoolExecutor(max_workers=COPY_THREADS) as ex:
    futures = [ex.submit(process_one, img) for img in all_images]
    for f in tqdm(as_completed(futures), total=len(futures), desc='  copy+convert'):
        split, n = f.result()
        stats[split][0 if n > 0 else 1] += 1

for split, (pos, neg) in stats.items():
    print(f'  {split:5s} → {pos} with targets | {neg} background')

# ── Write dataset.yaml ─────────────────────────────────────────────────────
print('[2/3] Writing dataset.yaml ...')
yaml_path = OUTPUT_DIR / 'dataset.yaml'
yaml.dump({
    'path':  str(OUTPUT_DIR),
    'train': 'train/images',
    'val':   'val/images',
    'nc':    len(CLASS_NAMES),
    'names': CLASS_NAMES,
}, open(yaml_path, 'w'), default_flow_style=False)

# ── Train ──────────────────────────────────────────────────────────────────
print(f'[3/3] Starting training with {YOLO_MODEL} ...')
model = YOLO(YOLO_MODEL)
model.train(
    data=str(yaml_path),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    name='pothole_patching_v1',
    project='/content/runs/train',
    patience=20,
    save=True,
    exist_ok=True,
    plots=True,
    seed=RANDOM_SEED,
    device=0,
    amp=True,
    cache='disk',
    workers=2,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
)


In [ ]:
# ─── Cell 5: Save best model to Google Drive ─────────────────────────────────
import shutil
from pathlib import Path

best_src = Path('/content/runs/train/pothole_patching_v1/weights/best.pt')
best_dst = Path('/content/drive/MyDrive/pothole_patching_best.pt')

shutil.copy2(best_src, best_dst)
print(f'Saved to Google Drive → {best_dst.name}')
print('Download it from your Google Drive!')